In [4]:
from langchain_community.utilities import SQLDatabase

In [12]:
db=SQLDatabase.from_uri("sqlite:///mydb.db")

In [13]:
db

In [14]:
db.dialect
#dialect is the type of database, in this case sqlite

'sqlite'

In [15]:
db.get_usable_table_names()

['customers', 'employees', 'orders']

In [17]:
from langchain_groq import ChatGroq

In [25]:
# Option A: Set it as an environment variable (Cleanest way)
#os.environ["GROQ_API_KEY"] = "your_actual_key_here"

In [21]:
llm=ChatGroq(model="llama-3.1-8b-instant")

#else use a different model, e.g. "gpt-4o"
#llm = ChatGroq(model="llama-3.3-70b-versatile")

In [22]:
llm.invoke("hello how are you?")

AIMessage(content="Hello, I'm just a language model, so I don't have emotions or feelings like humans do, but I'm functioning properly and ready to help you with any questions or tasks you may have. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 40, 'total_tokens': 88, 'completion_time': 0.056019703, 'completion_tokens_details': None, 'prompt_time': 0.002307763, 'prompt_tokens_details': None, 'queue_time': 0.052095694, 'total_time': 0.058327466}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1735-e044-7282-a9bb-161f5d81fd66-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 48, 'total_tokens': 88})

In [26]:
#SO LLM IS WORKING

In [31]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [32]:
toolkit=SQLDatabaseToolkit(db=db, llm=llm)

In [34]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 QuerySQLCheckerTool(description='Use this tool to 

In [35]:

tools=toolkit.get_tools()
tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>),
 QuerySQLCheckerTool(description='Use this tool to 

In [36]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [42]:
list_tables_tool = next((tool for tool in tools if tool.name == "sql_db_list_tables"), None)

In [43]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>)

In [44]:
get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"),None)

In [49]:
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E5BC3C4650>)

In [45]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [47]:
get_schema_tool.invoke("customers")

'\nCREATE TABLE customers (\n\tcustomer_id INTEGER, \n\tfirst_name TEXT NOT NULL, \n\tlast_name TEXT NOT NULL, \n\temail TEXT NOT NULL, \n\tphone TEXT, \n\tPRIMARY KEY (customer_id), \n\tUNIQUE (email)\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tfirst_name\tlast_name\temail\tphone\n1\tArjun\tMehta\tarjun.m@example.com\t9876543210\n2\tSara\tKhan\tsara.k@gmail.com\t9821098765\n3\tVikram\tIyer\tviyer@outlook.com\t9900887766\n*/'

In [48]:
#instead of this, we can print it cleanly
print(get_schema_tool.invoke("customers"))


CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
1	Arjun	Mehta	arjun.m@example.com	9876543210
2	Sara	Khan	sara.k@gmail.com	9821098765
3	Vikram	Iyer	viyer@outlook.com	9900887766
*/
